# Projekt 1 — świat klocków

Raport dla progu 8 punktów. Eksperymenty używają ziarna `42` i limitu `1 000 000` rozwiniętych węzłów. Przekroczenie limitu jest raportowane, a nie pomijane.

In [1]:
import json
from pathlib import Path
from statistics import mean
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == 'experiments' else Path.cwd()
RESULTS_PATH = ROOT / 'output' / 'results.json'
SEED = 42
MAX_EXPANDED = 1_000_000

assert RESULTS_PATH.exists(), f'Brak wymaganego pliku {RESULTS_PATH}'
records = json.loads(RESULTS_PATH.read_text(encoding='utf-8'))
assert len(records) == 135, f'Oczekiwano 135 rekordów, otrzymano {len(records)}'
invalid = [row['case_id'] for row in records if row['status'] == 'invalid_solution']
assert not invalid, f'Niepoprawne rozwiązania: {invalid}'
print(f'Wczytano {len(records)} rekordów z {RESULTS_PATH}')

Wczytano 135 rekordów z C:\Users\piotr\Projects\Blocks-World\output\results.json


In [2]:
def show_table(rows, columns):
    if not rows:
        display(Markdown('_Brak wyników dla tej sekcji._'))
        return
    header = '| ' + ' | '.join(columns) + ' |'
    separator = '| ' + ' | '.join('---' for _ in columns) + ' |'
    body = [
        '| ' + ' | '.join(str(row.get(column, '')) for column in columns) + ' |'
        for row in rows
    ]
    display(Markdown('\n'.join([header, separator, *body])))

def verified(rows):
    return [row for row in rows if row['verification'] == 'passed']

## Reprezentacja i kontrola poprawności

Stan jest niemutowalną krotką. Element na pozycji `i` określa bezpośrednie podłoże klocka `i+1`, a `0` oznacza stół. Walidator odrzuca nieistniejące podłoża, samopodparcie, dwa klocki na jednym podłożu i cykle. Każda sekwencja zwrócona przez solver jest ponownie wykonywana przez niezależny weryfikator, który wskazuje pierwszy niepoprawny ruch.

## Wariant podstawowy — DFS

DFS z nawrotami pomija stany już odwiedzone i kończy pracę po osiągnięciu limitu. Ruchy są rozpatrywane w deterministycznej kolejności generatora: ruchy na stół według numeru klocka, a następnie ruchy na klocek leksykograficznie po `(klocek, cel)`. Jest to kolejność z implementacji DFS kolegi; DFS zwraca pierwsze znalezione rozwiązanie i nie gwarantuje jego minimalnej długości.

In [3]:
dfs_basic = [
    row for row in records
    if row['algorithm'] == 'dfs' and row['variant'] == 'basic'
]
show_table(
    dfs_basic,
    ['case_id', 'n', 'start', 'goal', 'status', 'move_count', 'expanded_nodes', 'elapsed_seconds', 'verification'],
)

small = [row for row in verified(dfs_basic) if row['n'] <= 8]
for row in small:
    display(Markdown(f"**{row['case_id']}**: `{row['moves']}`"))

| case_id | n | start | goal | status | move_count | expanded_nodes | elapsed_seconds | verification |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| problem_n03_01 | 3 | [2, 0, 0] | [3, 0, 0] | solved | 2 | 2 | 4.3800013372674584e-05 | passed |
| problem_n03_02 | 3 | [0, 0, 0] | [0, 0, 2] | solved | 1 | 1 | 5.0799979362636805e-05 | passed |
| problem_n03_03 | 3 | [0, 3, 0] | [0, 0, 0] | solved | 1 | 1 | 4.290000651963055e-05 | passed |
| problem_n03_04 | 3 | [3, 0, 0] | [0, 3, 0] | solved | 2 | 2 | 3.060000017285347e-05 | passed |
| problem_n03_05 | 3 | [0, 1, 0] | [0, 0, 0] | solved | 1 | 1 | 1.6000005416572094e-05 | passed |
| problem_n04_01 | 4 | [0, 1, 0, 3] | [0, 1, 2, 0] | solved | 2 | 2 | 2.759997732937336e-05 | passed |
| problem_n04_02 | 4 | [0, 0, 4, 0] | [0, 0, 0, 0] | solved | 1 | 1 | 2.0100007532164454e-05 | passed |
| problem_n04_03 | 4 | [4, 0, 0, 3] | [0, 1, 0, 0] | solved | 3 | 3 | 4.7900015488266945e-05 | passed |
| problem_n04_04 | 4 | [0, 3, 4, 1] | [2, 0, 0, 1] | solved | 5 | 5 | 5.2400020649656653e-05 | passed |
| problem_n04_05 | 4 | [0, 1, 2, 0] | [2, 0, 0, 0] | solved | 3 | 3 | 3.9099977584555745e-05 | passed |
| problem_n05_01 | 5 | [4, 0, 5, 0, 0] | [5, 4, 0, 0, 0] | solved | 4 | 4 | 7.070001447573304e-05 | passed |
| problem_n05_02 | 5 | [4, 0, 0, 3, 2] | [0, 4, 0, 1, 0] | solved | 15 | 19 | 0.0001779999874997884 | passed |
| problem_n05_03 | 5 | [2, 0, 0, 0, 0] | [5, 4, 0, 3, 0] | solved | 4 | 4 | 5.329999839887023e-05 | passed |
| problem_n05_04 | 5 | [0, 1, 0, 0, 0] | [0, 0, 5, 2, 0] | solved | 3 | 3 | 6.120000034570694e-05 | passed |
| problem_n05_05 | 5 | [5, 0, 0, 0, 4] | [5, 1, 0, 0, 0] | solved | 6 | 8 | 7.919999188743532e-05 | passed |
| problem_n06_01 | 6 | [2, 3, 0, 6, 0, 0] | [5, 0, 0, 2, 6, 0] | solved | 8 | 8 | 0.0001244999875780195 | passed |
| problem_n06_02 | 6 | [5, 0, 2, 0, 6, 4] | [3, 0, 0, 6, 0, 0] | solved | 6 | 6 | 0.00011059999815188348 | passed |
| problem_n06_03 | 6 | [6, 0, 2, 3, 0, 0] | [0, 0, 0, 2, 1, 0] | solved | 5 | 5 | 7.069998537190259e-05 | passed |
| problem_n06_04 | 6 | [2, 0, 0, 6, 3, 0] | [0, 0, 1, 0, 0, 0] | solved | 4 | 4 | 8.189998334273696e-05 | passed |
| problem_n06_05 | 6 | [0, 4, 1, 0, 2, 3] | [0, 0, 0, 1, 6, 2] | solved | 13 | 13 | 0.00016029999824240804 | passed |
| problem_n07_01 | 7 | [0, 0, 7, 1, 0, 3, 0] | [3, 0, 0, 7, 6, 0, 0] | solved | 6 | 6 | 0.00013669999316334724 | passed |
| problem_n07_02 | 7 | [0, 0, 6, 1, 4, 0, 3] | [5, 0, 4, 6, 7, 0, 0] | solved | 16 | 16 | 0.0002798000059556216 | passed |
| problem_n07_03 | 7 | [0, 4, 7, 5, 3, 0, 0] | [4, 7, 2, 6, 0, 0, 1] | solved | 23 | 25 | 0.00035760001628659666 | passed |
| problem_n07_04 | 7 | [6, 4, 1, 5, 0, 7, 0] | [3, 7, 0, 2, 0, 1, 0] | solved | 11 | 11 | 0.00021649998961947858 | passed |
| problem_n07_05 | 7 | [4, 3, 1, 0, 0, 7, 2] | [0, 0, 0, 5, 0, 7, 1] | solved | 16 | 16 | 0.0004867000097874552 | passed |
| problem_n08_01 | 8 | [8, 0, 2, 7, 1, 3, 0, 0] | [6, 5, 0, 8, 0, 0, 1, 0] | solved | 9 | 9 | 0.00020840001525357366 | passed |
| problem_n08_02 | 8 | [0, 8, 1, 0, 0, 5, 6, 3] | [7, 0, 0, 1, 0, 4, 5, 3] | solved | 44 | 44 | 0.002038400009041652 | passed |
| problem_n08_03 | 8 | [0, 8, 4, 0, 3, 2, 0, 0] | [5, 0, 2, 0, 8, 0, 4, 6] | solved | 25 | 25 | 0.0003181000065524131 | passed |
| problem_n08_04 | 8 | [3, 0, 0, 2, 0, 5, 1, 7] | [0, 6, 7, 0, 8, 5, 2, 0] | solved | 56 | 56 | 0.0007360999879892915 | passed |
| problem_n08_05 | 8 | [0, 4, 2, 7, 0, 0, 0, 6] | [0, 0, 0, 8, 1, 5, 2, 6] | solved | 43 | 43 | 0.0006580999761354178 | passed |
| problem_n09_01 | 9 | [3, 0, 9, 5, 1, 8, 0, 7, 0] | [7, 0, 5, 9, 0, 2, 8, 3, 6] | solved | 17 | 17 | 0.0002818999928422272 | passed |
| problem_n09_02 | 9 | [7, 1, 0, 6, 2, 5, 0, 3, 4] | [0, 0, 2, 0, 9, 7, 8, 5, 0] | solved | 18 | 18 | 0.00043230000301264226 | passed |
| problem_n09_03 | 9 | [3, 0, 0, 9, 4, 0, 8, 2, 1] | [0, 7, 0, 5, 0, 0, 1, 2, 6] | solved | 47 | 47 | 0.0009892999951262027 | passed |
| problem_n09_04 | 9 | [0, 8, 4, 0, 1, 7, 0, 5, 0] | [7, 0, 2, 0, 4, 0, 8, 9, 6] | solved | 23 | 23 | 0.0003090999962296337 | passed |
| problem_n09_05 | 9 | [7, 0, 0, 0, 0, 0, 3, 6, 2] | [0, 0, 1, 8, 0, 2, 9, 6, 3] | solved | 92 | 92 | 0.0019244000141043216 | passed |
| problem_n10_01 | 10 | [0, 6, 0, 10, 7, 8, 0, 9, 0, 1] | [0, 9, 0, 6, 7, 8, 10, 0, 0, 2] | solved | 80 | 80 | 0.002289199997903779 | passed |
| problem_n10_02 | 10 | [2, 0, 0, 1, 4, 7, 0, 0, 3, 0] | [7, 4, 0, 0, 2, 1, 3, 5, 10, 0] | solved | 84 | 84 | 0.0013107999984640628 | passed |
| problem_n10_03 | 10 | [3, 6, 10, 0, 2, 0, 9, 5, 4, 0] | [4, 0, 2, 0, 9, 10, 6, 3, 0, 0] | solved | 41 | 41 | 0.0010342999885324389 | passed |
| problem_n10_04 | 10 | [7, 8, 5, 10, 6, 0, 3, 0, 1, 0] | [4, 5, 8, 2, 6, 0, 1, 0, 7, 3] | solved | 27 | 27 | 0.0003549999964889139 | passed |
| problem_n10_05 | 10 | [0, 5, 6, 3, 8, 0, 2, 1, 10, 4] | [0, 7, 5, 0, 6, 0, 4, 2, 0, 9] | solved | 22 | 22 | 0.000467099976958707 | passed |
| problem_n11_01 | 11 | [0, 0, 0, 11, 1, 0, 2, 0, 5, 7, 0] | [0, 0, 11, 0, 1, 2, 9, 3, 0, 4, 6] | solved | 32 | 32 | 0.0008580999856349081 | passed |
| problem_n11_02 | 11 | [5, 9, 2, 8, 0, 0, 0, 10, 6, 7, 3] | [0, 9, 1, 3, 6, 10, 5, 11, 0, 0, 0] | solved | 181 | 183 | 0.005586599989328533 | passed |
| problem_n11_03 | 11 | [11, 8, 2, 3, 0, 5, 0, 7, 0, 4, 0] | [0, 8, 11, 0, 2, 5, 10, 1, 0, 0, 7] | solved | 82 | 82 | 0.0020986000017728657 | passed |
| problem_n11_04 | 11 | [0, 3, 8, 0, 2, 10, 0, 0, 6, 11, 0] | [10, 11, 4, 9, 0, 0, 0, 7, 2, 5, 8] | solved | 168 | 168 | 0.003436600003624335 | passed |
| problem_n11_05 | 11 | [6, 0, 0, 8, 7, 5, 0, 0, 3, 0, 10] | [3, 4, 5, 0, 0, 8, 6, 0, 0, 2, 7] | solved | 15 | 15 | 0.0004253000079188496 | passed |
| problem_n12_01 | 12 | [0, 8, 12, 0, 7, 0, 0, 10, 3, 4, 6, 0] | [3, 4, 11, 7, 1, 12, 0, 0, 0, 2, 6, 0] | solved | 241 | 241 | 0.007551899994723499 | passed |
| problem_n12_02 | 12 | [0, 12, 5, 0, 6, 4, 0, 10, 8, 11, 7, 1] | [0, 9, 0, 3, 0, 4, 2, 0, 0, 0, 10, 1] | solved | 13 | 13 | 0.00046480001765303314 | passed |
| problem_n12_03 | 12 | [8, 1, 5, 3, 0, 0, 0, 12, 11, 2, 4, 0] | [3, 0, 9, 11, 10, 12, 0, 6, 0, 8, 1, 0] | solved | 102 | 102 | 0.0029474999755620956 | passed |
| problem_n12_04 | 12 | [8, 6, 4, 0, 0, 9, 10, 0, 5, 0, 1, 0] | [0, 0, 0, 0, 0, 4, 9, 2, 6, 3, 1, 11] | solved | 526 | 526 | 0.016476599994348362 | passed |
| problem_n12_05 | 12 | [0, 4, 0, 10, 6, 0, 9, 0, 11, 0, 3, 7] | [4, 0, 0, 9, 0, 0, 8, 6, 2, 5, 1, 3] | solved | 343 | 343 | 0.008268299978226423 | passed |
| problem_n13_01 | 13 | [0, 9, 4, 12, 8, 3, 1, 0, 11, 0, 7, 0, 5] | [5, 0, 4, 1, 8, 0, 0, 9, 0, 7, 0, 0, 3] | solved | 318 | 318 | 0.017825899994932115 | passed |
| problem_n13_02 | 13 | [0, 0, 0, 12, 0, 1, 0, 3, 4, 8, 6, 0, 9] | [0, 0, 7, 5, 1, 0, 6, 13, 3, 2, 4, 11, 0] | solved | 6922 | 6922 | 0.2883597999752965 | passed |
| problem_n13_03 | 13 | [0, 12, 2, 1, 3, 7, 4, 0, 10, 13, 9, 0, 8] | [7, 13, 0, 0, 6, 12, 0, 0, 1, 4, 8, 11, 9] | solved | 235 | 235 | 0.0058422999863978475 | passed |
| problem_n13_04 | 13 | [0, 0, 0, 5, 8, 12, 1, 10, 13, 7, 6, 0, 0] | [0, 12, 11, 1, 0, 0, 10, 9, 6, 0, 7, 0, 4] | solved | 62 | 62 | 0.0024127000069711357 | passed |
| problem_n13_05 | 13 | [10, 4, 9, 5, 0, 3, 0, 7, 1, 0, 8, 0, 0] | [3, 1, 0, 12, 13, 0, 6, 11, 8, 0, 0, 9, 10] | solved | 711 | 711 | 0.01767560001462698 | passed |
| problem_n14_01 | 14 | [10, 6, 5, 7, 0, 0, 0, 0, 4, 3, 14, 13, 0, 1] | [6, 0, 13, 2, 0, 4, 5, 9, 7, 0, 14, 10, 0, 0] | solved | 182 | 182 | 0.00555309999617748 | passed |
| problem_n14_02 | 14 | [0, 0, 0, 3, 10, 5, 8, 9, 11, 4, 0, 13, 1, 0] | [2, 0, 12, 13, 4, 0, 9, 1, 0, 0, 6, 11, 0, 0] | solved | 19 | 19 | 0.0010087999980896711 | passed |
| problem_n14_03 | 14 | [8, 9, 0, 0, 2, 1, 3, 5, 10, 0, 6, 7, 0, 0] | [9, 1, 0, 11, 4, 3, 6, 7, 10, 0, 0, 2, 8, 12] | solved | 1456 | 1456 | 0.03825660000438802 | passed |
| problem_n14_04 | 14 | [7, 3, 0, 1, 10, 5, 12, 6, 0, 14, 9, 11, 8, 0] | [4, 9, 5, 0, 14, 10, 13, 0, 3, 1, 7, 6, 0, 0] | solved | 783 | 783 | 0.030945300008170307 | passed |
| problem_n14_05 | 14 | [0, 10, 0, 0, 2, 4, 0, 7, 5, 14, 3, 13, 6, 0] | [0, 3, 13, 7, 8, 2, 10, 9, 0, 6, 0, 1, 0, 4] | solved | 3173 | 3173 | 0.1435217999969609 | passed |
| problem_n15_01 | 15 | [9, 14, 13, 6, 3, 11, 0, 15, 10, 12, 0, 0, 7, 0, 0] | [5, 15, 0, 10, 9, 1, 0, 3, 11, 12, 0, 0, 2, 13, 7] | solved | 2217 | 2217 | 0.06172210001386702 | passed |
| problem_n15_02 | 15 | [0, 7, 9, 6, 14, 8, 0, 0, 1, 15, 13, 4, 2, 3, 0] | [13, 0, 7, 1, 8, 10, 0, 9, 11, 4, 14, 15, 0, 2, 0] | solved | 2540 | 2540 | 0.10287719999905676 | passed |
| problem_n15_03 | 15 | [5, 8, 11, 1, 0, 0, 6, 14, 15, 7, 12, 13, 2, 0, 0] | [0, 0, 15, 5, 2, 8, 0, 10, 7, 11, 0, 1, 14, 12, 13] | solved | 6900 | 6900 | 0.262213400012115 | passed |
| problem_n15_04 | 15 | [10, 0, 12, 0, 1, 8, 13, 2, 6, 0, 15, 4, 0, 0, 14] | [9, 0, 0, 7, 0, 15, 6, 12, 13, 0, 10, 0, 14, 5, 3] | solved | 32 | 32 | 0.0009026999759953469 | passed |
| problem_n15_05 | 15 | [0, 0, 14, 0, 6, 13, 11, 9, 0, 12, 0, 2, 4, 10, 1] | [0, 0, 8, 9, 4, 13, 14, 0, 1, 7, 3, 15, 11, 2, 0] | solved | 223 | 223 | 0.005780100007541478 | passed |

**problem_n03_01**: `[[1, 0], [1, 3]]`

**problem_n03_02**: `[[3, 2]]`

**problem_n03_03**: `[[2, 0]]`

**problem_n03_04**: `[[1, 0], [2, 3]]`

...

**problem_n08_01**: `[[5, 0], [6, 0], [3, 0], [2, 5], [1, 0], [1, 6], [4, 0], [4, 8], [7, 1]]`

**problem_n08_02**: `[[2, 0], [7, 0], [6, 0], [6, 4], [7, 5], [6, 0], [4, 2], [6, 4], [7, 0], [7, 6], [8, 0], [3, 0], [1, 7], [8, 3], [1, 0], [7, 0], [1, 7], [6, 0], [4, 0], [4, 1], [6, 4], [8, 0], [8, 2], [6, 0], [8, 0], [6, 2], [8, 3], [4, 0], [4, 5], [6, 0], [6, 4], [1, 0], [1, 2], [7, 1], [6, 0], [4, 0], [6, 4], [7, 0], [7, 5], [1, 0], [1, 7], [6, 0], [4, 1], [6, 4]]`

**problem_n08_03**: `[[6, 0], [2, 0], [1, 5], [8, 6], [7, 1], [8, 0], [8, 2], [7, 0], [7, 6], [8, 0], [8, 1], [7, 0], [7, 2], [8, 0], [8, 6], [1, 0], [5, 0], [1, 5], [3, 0], [7, 0], [3, 2], [7, 4], [1, 0], [5, 8], [1, 5]]`

**problem_n08_04**: `[[4, 0], [8, 0], [2, 6], [7, 0], [1, 0], [3, 7], [2, 0], [2, 1], [3, 0], [7, 2], [3, 7], [6, 0], [5, 8], [6, 5], [3, 0], [3, 4], [6, 0], [3, 0], [3, 5], [6, 3], [7, 0], [2, 0], [2, 6], [7, 2], [1, 4], [7, 0], [7, 1], [2, 0], [6, 0], [2, 6], [3, 0], [3, 7], [2, 0], [6, 5], [2, 6], [3, 0], [7, 0], [1, 0], [3, 7], [2, 0], [2, 1], [3, 0], [2, 0], [7, 2], [3, 7], [6, 0], [6, 1], [3, 0], [6, 0], [3, 1], [6, 5], [7, 0], [2, 6], [7, 2], [3, 0], [3, 7]]`

**problem_n08_05**: `[[3, 0], [2, 0], [5, 1], [4, 0], [4, 8], [7, 2], [4, 0], [4, 3], [5, 0], [4, 0], [4, 8], [5, 3], [4, 0], [4, 1], [5, 0], [5, 4], [7, 0], [5, 0], [4, 0], [4, 8], [5, 2], [7, 1], [5, 0], [5, 3], [7, 0], [7, 4], [5, 0], [5, 1], [2, 3], [7, 0], [7, 2], [4, 0], [4, 5], [7, 0], [2, 0], [7, 2], [8, 0], [4, 0], [4, 8], [6, 5], [4, 0], [8, 6], [4, 8]]`

## Wariant rozszerzony — DFS kontra BFS

BFS gwarantuje najkrótszą sekwencję, natomiast DFS zwraca pierwsze znalezione rozwiązanie. Porównanie wykorzystuje te same 10 problemów z `N=8`; rzeczywiste różnice pamięci, czasu i liczby węzłów są wyliczane poniżej.

In [4]:
comparison = [
    row for row in records
    if row['variant'] == 'extended'
    and row['n'] == 8
    and row['algorithm'] in {'dfs', 'bfs'}
]
show_table(
    comparison,
    ['case_id', 'algorithm', 'status', 'move_count', 'expanded_nodes', 'peak_stored_states', 'elapsed_seconds', 'verification'],
)
assert len(comparison) == 20, f'Oczekiwano 20 wyników DFS/BFS, otrzymano {len(comparison)}'
assert all(row['peak_stored_states'] is not None for row in comparison), 'Brak pomiaru pamięci'
comparison_summary = {}
for algorithm in ('dfs', 'bfs'):
    rows = verified([row for row in comparison if row['algorithm'] == algorithm])
    assert len(rows) == 10, f'{algorithm}: nie wszystkie wyniki są poprawne'
    comparison_summary[algorithm] = {
        'algorithm': algorithm.upper(),
        'solved': len(rows),
        'mean_move_count': round(mean(row['move_count'] for row in rows), 3),
        'mean_expanded_nodes': round(mean(row['expanded_nodes'] for row in rows), 3),
        'mean_peak_stored_states': round(mean(row['peak_stored_states'] for row in rows), 3),
        'mean_elapsed_seconds': round(mean(row['elapsed_seconds'] for row in rows), 6),
    }
show_table(list(comparison_summary.values()), list(next(iter(comparison_summary.values()))))
pairs = {}
for row in comparison:
    pairs.setdefault(row['case_id'], {})[row['algorithm']] = row
assert all(pair['bfs']['move_count'] <= pair['dfs']['move_count'] for pair in pairs.values())
strictly_shorter = sum(pair['bfs']['move_count'] < pair['dfs']['move_count'] for pair in pairs.values())
display(Markdown(
    f'BFS znalazł krótszą sekwencję w **{strictly_shorter} z 10** przypadków. '
    f'Średnia długość wyniosła **{comparison_summary["bfs"]["mean_move_count"]}** dla BFS '
    f'i **{comparison_summary["dfs"]["mean_move_count"]}** dla DFS.'
))

| case_id | algorithm | status | move_count | expanded_nodes | peak_stored_states | elapsed_seconds | verification |
| --- | --- | --- | --- | --- | --- | --- | --- |
| problem_n08_01 | dfs | solved | 8 | 8 | 9 | 0.0002248000237159431 | passed |
| problem_n08_02 | dfs | solved | 15 | 15 | 16 | 0.00046779998228885233 | passed |
| problem_n08_03 | dfs | solved | 7 | 7 | 8 | 0.00021669999114237726 | passed |
| problem_n08_04 | dfs | solved | 9 | 9 | 10 | 0.0001790999958757311 | passed |
| problem_n08_05 | dfs | solved | 7 | 7 | 8 | 0.00025279997498728335 | passed |
| problem_n08_06 | dfs | solved | 71 | 71 | 72 | 0.0013769000070169568 | passed |
| problem_n08_07 | dfs | solved | 10 | 10 | 11 | 0.00039280002238228917 | passed |
| problem_n08_08 | dfs | solved | 20 | 20 | 21 | 0.00042910000775009394 | passed |
| problem_n08_09 | dfs | solved | 8 | 8 | 9 | 0.0002764999808277935 | passed |
| problem_n08_10 | dfs | solved | 22 | 22 | 23 | 0.00044460000935941935 | passed |
| problem_n08_01 | bfs | solved | 6 | 21715 | 81917 | 0.3054634000000078 | passed |
| problem_n08_02 | bfs | solved | 6 | 2298 | 11881 | 0.03070470000966452 | passed |
| problem_n08_03 | bfs | solved | 6 | 4174 | 18938 | 0.05285780000849627 | passed |
| problem_n08_04 | bfs | solved | 7 | 24735 | 80246 | 0.3124291000131052 | passed |
| problem_n08_05 | bfs | solved | 7 | 27452 | 91246 | 0.3697094999952242 | passed |
| problem_n08_06 | bfs | solved | 9 | 107183 | 227040 | 1.54879030000302 | passed |
| problem_n08_07 | bfs | solved | 8 | 136890 | 262956 | 1.9300707999791484 | passed |
| problem_n08_08 | bfs | solved | 8 | 61732 | 156340 | 1.0134449999895878 | passed |
| problem_n08_09 | bfs | solved | 8 | 26295 | 84731 | 0.39103340002475306 | passed |
| problem_n08_10 | bfs | solved | 7 | 107662 | 236062 | 1.789411999983713 | passed |

| algorithm | solved | mean_move_count | mean_expanded_nodes | mean_peak_stored_states | mean_elapsed_seconds |
| --- | --- | --- | --- | --- | --- |
| DFS | 10 | 17.7 | 17.7 | 18.7 | 0.000426 |
| BFS | 10 | 7.2 | 52013.6 | 125135.7 | 0.774392 |

BFS znalazł krótszą sekwencję w **8 z 10** przypadków. Średnia długość wyniosła **7.2** dla BFS i **17.7** dla DFS.

## IDA* i heurystyki

Pierwsza heurystyka `misplaced_supports` liczy klocki leżące na innym bezpośrednim podłożu niż w celu. Jest dopuszczalna, ponieważ pojedynczy ruch zmienia podłoże dokładnie jednego klocka, więc może usunąć najwyżej jedną taką niezgodność.

Druga heurystyka `misplaced_supports_with_blockers` dodaje po dwa ruchy za każdy poprawnie ustawiony klocek znajdujący się nad klockiem o błędnym podłożu. Taki klocek trzeba co najmniej raz zdjąć, aby odsłonić błędnie ustawiony klocek, a następnie co najmniej raz odłożyć na jego niezmienione docelowe podłoże. Ruchy te dotyczą innych klocków niż ruchy zliczone przez pierwszą heurystykę, więc suma nadal nie przekracza kosztu rozwiązania. Druga heurystyka dominuje pierwszą.

In [5]:
ida = [row for row in records if row['algorithm'] == 'ida_star']
show_table(
    ida,
    ['case_id', 'n', 'heuristic', 'status', 'move_count', 'expanded_nodes', 'elapsed_seconds', 'verification'],
)

assert len(ida) == 50, f'Oczekiwano 50 wyników IDA*, otrzymano {len(ida)}'
expected_heuristics = {'misplaced_supports', 'misplaced_supports_with_blockers'}
assert {row['heuristic'] for row in ida} == expected_heuristics
ida_summary = []
for heuristic in sorted(expected_heuristics):
    for n in (6, 8, 10, 12, 15):
        rows = [row for row in ida if row['heuristic'] == heuristic and row['n'] == n]
        assert len(rows) == 5
        ida_summary.append({
            'heuristic': heuristic,
            'n': n,
            'solved': sum(row['status'] == 'solved' for row in rows),
            'node_limit': sum(row['status'] == 'node_limit' for row in rows),
            'mean_expanded_nodes': round(mean(row['expanded_nodes'] for row in rows), 3),
            'mean_elapsed_seconds': round(mean(row['elapsed_seconds'] for row in rows), 6),
        })
show_table(ida_summary, ['heuristic', 'n', 'solved', 'node_limit', 'mean_expanded_nodes', 'mean_elapsed_seconds'])

by_case = {}
for row in ida:
    by_case.setdefault(row['case_id'], []).append(row)
comparable = {}
skipped = []
for case_id, rows in by_case.items():
    solved = {row['heuristic']: row for row in verified(rows)}
    if len(expected_heuristics) >= 2 and set(solved) == expected_heuristics:
        comparable[case_id] = solved
    else:
        skipped.append(case_id)
mismatches = {
    case_id: sorted({row['move_count'] for row in rows.values()})
    for case_id, rows in comparable.items()
    if len({row['move_count'] for row in rows.values()}) > 1
}
assert not mismatches, f'Heurystyki zwróciły różne długości: {mismatches}'
assert comparable, 'Brak kompletnych par wyników IDA* do porównania'
print(f'Sprawdzenie zgodności długości IDA*: OK ({len(comparable)} porównanych par)')
if skipped:
    print(f'Pominięte przypadki (limit lub brak wyniku jednej heurystyki): {len(skipped)}')
improved_better = sum(
    rows['misplaced_supports_with_blockers']['expanded_nodes']
    < rows['misplaced_supports']['expanded_nodes']
    for rows in comparable.values()
)
display(Markdown(
    f'Wśród **{len(comparable)}** kompletnych par silniejsza heurystyka rozwinęła mniej węzłów '
    f'w **{improved_better}** przypadkach. Dla pozostałych przypadków limit jest jawnie pokazany w tabeli.'
))

| case_id | n | heuristic | status | move_count | expanded_nodes | elapsed_seconds | verification |
| --- | --- | --- | --- | --- | --- | --- | --- |
| problem_n06_01 | 6 | misplaced_supports | solved | 5 | 13 | 0.0002350000140722841 | passed |
| problem_n06_02 | 6 | misplaced_supports | solved | 5 | 5 | 0.00010989999282173812 | passed |
| problem_n06_03 | 6 | misplaced_supports | solved | 5 | 9 | 0.00031939998734742403 | passed |
| problem_n06_04 | 6 | misplaced_supports | solved | 4 | 4 | 7.080001523718238e-05 | passed |
| problem_n06_05 | 6 | misplaced_supports | solved | 6 | 10 | 0.0001393999846186489 | passed |
| problem_n08_01 | 8 | misplaced_supports | solved | 7 | 7 | 0.00015810001059435308 | passed |
| problem_n08_02 | 8 | misplaced_supports | solved | 10 | 8788 | 0.20793020000564866 | passed |
| problem_n08_03 | 8 | misplaced_supports | solved | 7 | 19 | 0.0006313000048976392 | passed |
| problem_n08_04 | 8 | misplaced_supports | solved | 9 | 783 | 0.03446249998523854 | passed |
| problem_n08_05 | 8 | misplaced_supports | solved | 8 | 857 | 0.0349825999874156 | passed |
| problem_n10_01 | 10 | misplaced_supports | solved | 10 | 93985 | 3.9652698000136297 | passed |
| problem_n10_02 | 10 | misplaced_supports | solved | 10 | 774 | 0.032815400016261265 | passed |
| problem_n10_03 | 10 | misplaced_supports | solved | 10 | 228 | 0.012096600024960935 | passed |
| problem_n10_04 | 10 | misplaced_supports | solved | 11 | 28430 | 0.6220560999936424 | passed |
| problem_n10_05 | 10 | misplaced_supports | solved | 9 | 103 | 0.0024234000011347234 | passed |
| problem_n12_01 | 12 | misplaced_supports | node_limit | None | 1000000 | 88.35228779999306 | not_applicable |
| problem_n12_02 | 12 | misplaced_supports | solved | 11 | 87249 | 8.298097199993208 | passed |
| problem_n12_03 | 12 | misplaced_supports | node_limit | None | 1000000 | 80.46595480001997 | not_applicable |
| problem_n12_04 | 12 | misplaced_supports | solved | 11 | 16508 | 1.1259522000036668 | passed |
| problem_n12_05 | 12 | misplaced_supports | solved | 11 | 143 | 0.008934999990742654 | passed |
| problem_n15_01 | 15 | misplaced_supports | node_limit | None | 1000000 | 86.06654820000404 | not_applicable |
| problem_n15_02 | 15 | misplaced_supports | solved | 16 | 3488 | 0.2454810999915935 | passed |
| problem_n15_03 | 15 | misplaced_supports | solved | 18 | 78641 | 6.908757900004275 | passed |
| problem_n15_04 | 15 | misplaced_supports | solved | 15 | 98815 | 9.358697600022424 | passed |
| problem_n15_05 | 15 | misplaced_supports | node_limit | None | 1000000 | 74.16669250000268 | not_applicable |
| problem_n06_01 | 6 | misplaced_supports_with_blockers | solved | 5 | 6 | 0.00014899999951012433 | passed |
| problem_n06_02 | 6 | misplaced_supports_with_blockers | solved | 5 | 5 | 9.580000187270343e-05 | passed |
| problem_n06_03 | 6 | misplaced_supports_with_blockers | solved | 5 | 8 | 0.0002634999982547015 | passed |
| problem_n06_04 | 6 | misplaced_supports_with_blockers | solved | 4 | 4 | 8.399999933317304e-05 | passed |
| problem_n06_05 | 6 | misplaced_supports_with_blockers | solved | 6 | 7 | 0.0002714000002015382 | passed |
| problem_n08_01 | 8 | misplaced_supports_with_blockers | solved | 7 | 7 | 0.00019540000357665122 | passed |
| problem_n08_02 | 8 | misplaced_supports_with_blockers | solved | 10 | 480 | 0.03117309999652207 | passed |
| problem_n08_03 | 8 | misplaced_supports_with_blockers | solved | 7 | 7 | 0.00021719999494962394 | passed |
| problem_n08_04 | 8 | misplaced_supports_with_blockers | solved | 9 | 9 | 0.0006292999896686524 | passed |
| problem_n08_05 | 8 | misplaced_supports_with_blockers | solved | 8 | 8 | 0.0002717999741435051 | passed |
| problem_n10_01 | 10 | misplaced_supports_with_blockers | solved | 10 | 15 | 0.0005770999996457249 | passed |
| problem_n10_02 | 10 | misplaced_supports_with_blockers | solved | 10 | 92 | 0.006638199993176386 | passed |
| problem_n10_03 | 10 | misplaced_supports_with_blockers | solved | 10 | 50 | 0.002256800013128668 | passed |
| problem_n10_04 | 10 | misplaced_supports_with_blockers | solved | 11 | 1011 | 0.03604500001529232 | passed |
| problem_n10_05 | 10 | misplaced_supports_with_blockers | solved | 9 | 51 | 0.0019319999846629798 | passed |
| problem_n12_01 | 12 | misplaced_supports_with_blockers | solved | 14 | 134061 | 14.047549799986882 | passed |
| problem_n12_02 | 12 | misplaced_supports_with_blockers | solved | 11 | 11 | 0.0008608000061940402 | passed |
| problem_n12_03 | 12 | misplaced_supports_with_blockers | solved | 14 | 36858 | 3.4617740000248887 | passed |
| problem_n12_04 | 12 | misplaced_supports_with_blockers | solved | 11 | 11 | 0.0010324999748263508 | passed |
| problem_n12_05 | 12 | misplaced_supports_with_blockers | solved | 11 | 31 | 0.0034558000043034554 | passed |
| problem_n15_01 | 15 | misplaced_supports_with_blockers | solved | 15 | 91390 | 10.751730999996653 | passed |
| problem_n15_02 | 15 | misplaced_supports_with_blockers | solved | 16 | 1298 | 0.13745310000376776 | passed |
| problem_n15_03 | 15 | misplaced_supports_with_blockers | solved | 18 | 3501 | 0.48785740000312217 | passed |
| problem_n15_04 | 15 | misplaced_supports_with_blockers | solved | 15 | 1213 | 0.19454230001429096 | passed |
| problem_n15_05 | 15 | misplaced_supports_with_blockers | node_limit | None | 1000000 | 129.09877969999798 | not_applicable |

| heuristic | n | solved | node_limit | mean_expanded_nodes | mean_elapsed_seconds |
| --- | --- | --- | --- | --- | --- |
| misplaced_supports | 6 | 5 | 0 | 8.2 | 0.000175 |
| misplaced_supports | 8 | 5 | 0 | 2090.8 | 0.055633 |
| misplaced_supports | 10 | 5 | 0 | 24704 | 0.926932 |
| misplaced_supports | 12 | 3 | 2 | 420780 | 35.650245 |
| misplaced_supports | 15 | 3 | 2 | 436188.8 | 35.349235 |
| misplaced_supports_with_blockers | 6 | 5 | 0 | 6 | 0.000173 |
| misplaced_supports_with_blockers | 8 | 5 | 0 | 102.2 | 0.006497 |
| misplaced_supports_with_blockers | 10 | 5 | 0 | 243.8 | 0.00949 |
| misplaced_supports_with_blockers | 12 | 5 | 0 | 34194.4 | 3.502935 |
| misplaced_supports_with_blockers | 15 | 4 | 1 | 219480.4 | 28.134073 |

Sprawdzenie zgodności długości IDA*: OK (21 porównanych par)
Pominięte przypadki (limit lub brak wyniku jednej heurystyki): 4


Wśród **21** kompletnych par silniejsza heurystyka rozwinęła mniej węzłów w **18** przypadkach. Dla pozostałych przypadków limit jest jawnie pokazany w tabeli.

In [6]:
basic_solved = sum(row['status'] == 'solved' for row in dfs_basic)
ida_limits = sum(row['status'] == 'node_limit' for row in ida)
display(Markdown(
    '## Wnioski\n\n'
    f'DFS rozwiązał **{basic_solved} z 65** problemów wariantu podstawowego. Algorytm '
    'używa kolejności generatora ruchów i nie gwarantuje najkrótszej sekwencji. '
    f'W porównaniu dla `N=8` BFS miał średnią długość **{comparison_summary["bfs"]["mean_move_count"]}**, '
    f'a DFS **{comparison_summary["dfs"]["mean_move_count"]}**; koszt optymalności widać w liczbie '
    'rozwiniętych i przechowywanych stanów oraz czasie. '
    f'IDA* zakończył **{ida_limits} z 50** uruchomień limitem. Heurystyka uwzględniająca blokujące '
    'klocki dominuje prostą heurystykę i częściej ogranicza liczbę rozwinięć, zachowując tę samą '
    f'optymalną długość we wszystkich **{len(comparable)}** kompletnych parach.'
))

## Wnioski

DFS rozwiązał **65 z 65** problemów wariantu podstawowego. Porządkowanie ruchów kieruje wyszukiwanie ku stanowi docelowemu, ale DFS nadal nie gwarantuje najkrótszej sekwencji. W porównaniu dla `N=8` BFS miał średnią długość **7.2**, a DFS **17.7**; koszt optymalności widać w liczbie rozwiniętych i przechowywanych stanów oraz czasie. IDA* zakończył **5 z 50** uruchomień limitem. Heurystyka uwzględniająca blokujące klocki dominuje prostą heurystykę i częściej ogranicza liczbę rozwinięć, zachowując tę samą optymalną długość we wszystkich **21** kompletnych parach.